# MedSpeak YO↔EN — MarianMT Fine-Tuning (Colab T4)

Trains **both** translation directions on a domain-balanced corpus (medical + general + greetings), with checkpoints written to Google Drive so a dropped session never costs more than one epoch.

**Before you run anything:** `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`.

Run the cells top to bottom. If the session disconnects, just reconnect and re-run the training cell — it auto-resumes from the last Drive checkpoint.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo
Pulls the latest code (including the domain-balanced training script) from GitHub.

In [ ]:
!git clone https://github.com/eddy-udoh/fyp-yoruba-english-s2st.git
%cd fyp-yoruba-english-s2st
!git log --oneline -1

## 3. Install training dependencies
Only what training needs — deliberately NOT `requirements.txt` (which pulls Coqui `TTS`/`librosa` that fight Colab's NumPy and aren't used here). `torch` ships with Colab.

In [ ]:
!pip install -q -U transformers datasets sacrebleu sentencepiece accelerate

## 4. Mount Google Drive (checkpoint persistence)
Checkpoints + the final model are written straight to Drive, so they survive a disconnect and you keep them after the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

YOEN_DIR = '/content/drive/MyDrive/marian-yoruba-medical'
ENYO_DIR = '/content/drive/MyDrive/marian-english-yoruba'
print('yo->en checkpoints ->', YOEN_DIR)
print('en->yo checkpoints ->', ENYO_DIR)

## 5. Train  yo → en  (Yorùbá → English)
~20–40 min on a T4. Watch the per-epoch **eval BLEU** and the **greeting sanity check** printed at the end.

If the session dropped mid-run, just re-run this cell — you'll see `[RESUME] Found checkpoint …`.

In [ ]:
!python src/nmt/marian_finetune.py --direction yo-en --output-dir "$YOEN_DIR"

## 6. Train  en → yo  (English → Yorùbá)
Same again for the other direction. The script auto-prefixes `>>yor<< ` and uses `opus-mt-en-nic` as the base.

In [ ]:
!python src/nmt/marian_finetune.py --direction en-yo --output-dir "$ENYO_DIR"

## 7. Save eval results to Drive & verify the models
The model weights are already on Drive (via `--output-dir`). This copies the per-direction eval JSONs too, then lists what was produced.

In [ ]:
!mkdir -p /content/drive/MyDrive/fyp_eval
!cp evaluation/nmt_finetuned_*.json /content/drive/MyDrive/fyp_eval/ 2>/dev/null; echo 'eval JSONs copied'
print('\n--- yo->en model files ---')
!ls -lh "$YOEN_DIR" | grep -E 'safetensors|config.json|spm|tokenizer'
print('\n--- en->yo model files ---')
!ls -lh "$ENYO_DIR" | grep -E 'safetensors|config.json|spm|tokenizer'

## 8. Integrate back into the repo (do this on your local machine)

The trained weights live on your Drive. The API ([`src/api/app.py`](../src/api/app.py)) loads each model from the **top-level** folder, so drop them straight in:

```
models/marian-yoruba-medical/      ← contents of Drive's marian-yoruba-medical/
models/marian-english-yoruba/      ← contents of Drive's marian-english-yoruba/
```

Steps locally:
1. Download both folders from Google Drive (`marian-yoruba-medical`, `marian-english-yoruba`).
2. Copy their **contents** (config.json, *.spm, tokenizer files, **model.safetensors**) into the matching `models/<dir>/` folders in the repo.
3. Each folder must contain `config.json`, `generation_config.json`, `source.spm`, `target.spm`, `tokenizer_config.json`, `vocab.json`, and `model.safetensors`.
4. Start the API — it logs `Loading fine-tuned en→yo model …` when it finds them.

> The weights are git-ignored (too large for GitHub), so they stay out of version control — keep the Drive copy as your backup.

**Then run the evaluation** (`src/eval/end_to_end_eval.py`, `src/eval/results_report.py`) on the machine that has the models + GPU.